<a href="https://colab.research.google.com/github/Vishu235/MetaBEARS/blob/main/colab/MetaBEARS_MiniKandinsky.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# MetaBEARS - MiniKandinsky Training and Evaluation

This notebook trains a reproducible MiniKandinsky baseline and evaluates the saved ensemble with MetaBEARS diagnostics, controlled semantic interventions, and a test-only OOD shift.

## Before running

1. Select **Runtime -> Change runtime type -> T4 GPU**.
2. Upload `kand-3k.zip` to:

`MyDrive/PES - Semester 4/metabears_data/kand-3k.zip`

The local source archive is available under the Phase I `bears/XOR_MNIST/data` folder. Existing checkpoints under `metabears_minikandinsky/seed_*` can be evaluated without retraining.


In [ ]:
#@title 1. Configuration
REPO_URL = "https://github.com/Vishu235/MetaBEARS.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
REPO_DIR = "/content/MetaBEARS"  #@param {type:"string"}

DRIVE_DATA_DIR = "/content/drive/MyDrive/PES - Semester 4/metabears_data"  #@param {type:"string"}
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/PES - Semester 4/metabears_minikandinsky"  #@param {type:"string"}
KAND_ZIP = f"{DRIVE_DATA_DIR}/kand-3k.zip"

RUN_SMOKE = False  #@param {type:"boolean"}
RUN_FULL_TRAINING = False  #@param {type:"boolean"}
RUN_METABEARS_SMOKE = False  #@param {type:"boolean"}
RUN_METABEARS_FULL = False  #@param {type:"boolean"}
SMOKE_EPOCHS = 1  #@param {type:"integer"}
FULL_EPOCHS = 30  #@param {type:"integer"}
BATCH_SIZE = 16  #@param {type:"integer"}
SEEDS = [0, 10, 20]

print("Configuration ready.")
print("Full training enabled:", RUN_FULL_TRAINING)
print("MetaBEARS smoke/full:", RUN_METABEARS_SMOKE, RUN_METABEARS_FULL)


In [ ]:
#@title 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
#@title 3. Clone or update MetaBEARS
import os
import subprocess
from pathlib import Path

repo = Path(REPO_DIR)
if repo.exists() and (repo / '.git').exists():
    subprocess.run(['git', 'fetch', 'origin'], cwd=repo, check=True)
    subprocess.run(['git', 'checkout', BRANCH], cwd=repo, check=True)
    subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=repo, check=True)
elif repo.exists():
    raise RuntimeError(f'{REPO_DIR} exists but is not a Git repository.')
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
subprocess.run(['git', 'log', '--oneline', '-3'], check=True)


In [ ]:
#@title 4. Install Colab-safe dependencies
import subprocess

subprocess.run(['python', '-m', 'pip', 'install', '-q', '--upgrade', 'pip'], check=True)
subprocess.run(['python', '-m', 'pip', 'install', '-q', '-r', 'requirements.colab.txt'], cwd=REPO_DIR, check=True)
print('Dependencies installed without replacing Colab CUDA PyTorch.')


In [ ]:
#@title 5. Runtime diagnostics
import subprocess
subprocess.run(['python', 'colab_runner.py', '--job', 'diagnostics'], cwd=REPO_DIR, check=True)


In [ ]:
#@title 6. Validate and stage kand-3k.zip
import hashlib
import shutil
import zipfile
from pathlib import Path

source_zip = Path(KAND_ZIP)
if not source_zip.exists():
    raise FileNotFoundError(f'Upload kand-3k.zip to {source_zip}')

def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

with zipfile.ZipFile(source_zip) as archive:
    corrupt = archive.testzip()
    if corrupt is not None:
        raise RuntimeError(f'Corrupt member in kand-3k.zip: {corrupt}')

repo_zip = Path(REPO_DIR) / 'XOR_MNIST' / 'data' / 'kand-3k.zip'
repo_zip.parent.mkdir(parents=True, exist_ok=True)
if not repo_zip.exists() or sha256(repo_zip) != sha256(source_zip):
    shutil.copy2(source_zip, repo_zip)

Path(DRIVE_RESULTS_DIR).mkdir(parents=True, exist_ok=True)
print('Dataset ready:', repo_zip)
print('SHA-256:', sha256(repo_zip))


In [ ]:
#@title 7. Reusable job helper
import shlex
import subprocess
from pathlib import Path

def run_job(job, *, seed, epochs, batch_size):
    command = [
        'python', 'colab_runner.py', '--job', job,
        '--seed', str(seed), '--epochs', str(epochs),
        '--batch-size', str(batch_size),
    ]
    print('\n>>>', ' '.join(shlex.quote(part) for part in command), flush=True)
    completed = subprocess.run(command, cwd=REPO_DIR, check=False)
    if completed.returncode:
        logs = sorted((Path(REPO_DIR) / 'logs').glob('*.log'), key=lambda p: p.stat().st_mtime)
        if logs:
            print(''.join(logs[-1].read_text(errors='replace').splitlines(True)[-160:]))
        raise SystemExit(f'{job} failed with exit code {completed.returncode}')
    return sorted((Path(REPO_DIR) / 'logs').glob('*.log'), key=lambda p: p.stat().st_mtime)[-1]

print('Job helper ready.')


In [ ]:
#@title 8. One-epoch MiniKandinsky smoke test
if RUN_SMOKE:
    smoke_log = run_job('minikand_smoke', seed=0, epochs=SMOKE_EPOCHS, batch_size=BATCH_SIZE)
    smoke_target = Path(DRIVE_RESULTS_DIR) / 'smoke' / smoke_log.name
    smoke_target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(smoke_log, smoke_target)
    print('Smoke test passed. Log saved to:', smoke_target)
else:
    print('Smoke test disabled.')


In [ ]:
#@title 9. Three-seed MiniKandinsky baseline training
import json
from datetime import datetime, timezone

if RUN_FULL_TRAINING:
    commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
    run_records = []
    for seed in SEEDS:
        log_path = run_job('minikand_train', seed=seed, epochs=FULL_EPOCHS, batch_size=BATCH_SIZE)
        checkpoint = Path(REPO_DIR) / 'XOR_MNIST' / 'data' / 'ckpts' / f'minikandinsky-minikanddpl-dis-{seed}-end.pt'
        if not checkpoint.exists():
            raise FileNotFoundError(checkpoint)
        seed_dir = Path(DRIVE_RESULTS_DIR) / f'seed_{seed}'
        seed_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy2(checkpoint, seed_dir / checkpoint.name)
        shutil.copy2(log_path, seed_dir / log_path.name)
        run_records.append({'seed': seed, 'checkpoint': checkpoint.name, 'log': log_path.name})

    manifest = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'git_commit': commit, 'dataset': 'minikandinsky',
        'model': 'minikanddpl', 'task': 'mini_patterns_bombazza',
        'epochs': FULL_EPOCHS, 'batch_size': BATCH_SIZE,
        'seeds': SEEDS, 'dataset_sha256': sha256(Path(KAND_ZIP)),
        'runs': run_records,
    }
    manifest_path = Path(DRIVE_RESULTS_DIR) / 'baseline_manifest.json'
    manifest_path.write_text(json.dumps(manifest, indent=2))
    print('Training complete. Manifest:', manifest_path)
else:
    print('Full training disabled. Enable it only after the smoke test passes.')


In [ ]:
#@title 10. Resolve the trained ensemble checkpoints
CHECKPOINT_PATHS = [
    str(Path(DRIVE_RESULTS_DIR) / f'seed_{seed}' / f'minikandinsky-minikanddpl-dis-{seed}-end.pt')
    for seed in SEEDS
]
if RUN_METABEARS_SMOKE or RUN_METABEARS_FULL:
    missing = [path for path in CHECKPOINT_PATHS if not Path(path).is_file()]
    if missing:
        raise FileNotFoundError('Missing trained checkpoints: ' + ', '.join(missing))
    print('Evaluation ensemble:')
    for checkpoint in CHECKPOINT_PATHS:
        print(' -', checkpoint)
else:
    print('MetaBEARS evaluation disabled in the Configuration cell.')


In [ ]:
#@title 11. Reusable MetaBEARS evaluation helper
import time

def run_metabears_evaluation(intervention, ood_transform, *, smoke=False):
    mode = 'smoke' if smoke else 'full'
    run_name = f"{time.strftime('%Y%m%d_%H%M%S')}_{intervention}_{mode}"
    output_dir = Path(DRIVE_RESULTS_DIR) / 'evaluations' / run_name
    command = [
        'python', 'colab_runner.py', '--job', 'metabears_minikandinsky',
        '--seed', '0', '--batch-size', str(BATCH_SIZE),
        '--minikand-output-dir', str(output_dir),
        '--minikand-intervention', intervention,
        '--minikand-ood-transform', ood_transform,
        '--minikand-checkpoints', *CHECKPOINT_PATHS,
    ]
    if smoke:
        command.extend(['--metabears-max-batches', '1'])
    print('\n>>>', ' '.join(shlex.quote(part) for part in command), flush=True)
    completed = subprocess.run(command, cwd=REPO_DIR, check=False)
    logs = sorted((Path(REPO_DIR) / 'logs').glob('*.log'), key=lambda path: path.stat().st_mtime)
    if completed.returncode:
        if logs:
            print(''.join(logs[-1].read_text(errors='replace').splitlines(True)[-200:]))
        raise SystemExit(f'MetaBEARS evaluation failed with exit code {completed.returncode}')
    summary_path = output_dir / 'run_summary.json'
    if not summary_path.is_file():
        raise FileNotFoundError(summary_path)
    if logs:
        shutil.copy2(logs[-1], output_dir / logs[-1].name)
    summary = json.loads(summary_path.read_text())
    print('Saved evaluation:', output_dir)
    print('ID task accuracy:', summary['splits']['id_test']['task_accuracy'])
    if summary.get('ood_detection') is not None:
        print('OOD AUROC:', summary['ood_detection']['auroc'])
    return output_dir, summary

print('MetaBEARS helper ready.')


In [ ]:
#@title 12. One-batch MetaBEARS integration check
if RUN_METABEARS_SMOKE:
    smoke_output, smoke_summary = run_metabears_evaluation(
        'figure_permute', 'palette_desaturate', smoke=True
    )
    print('MetaBEARS integration check passed:', smoke_output)
else:
    print('MetaBEARS integration check disabled.')


In [ ]:
#@title 13. Full MiniKandinsky MetaBEARS evaluation
if RUN_METABEARS_FULL:
    evaluation_records = []
    evaluation_matrix = [
        ('figure_permute', 'palette_desaturate'),
        ('palette_cycle', 'none'),
    ]
    for intervention, ood_transform in evaluation_matrix:
        output_dir, summary = run_metabears_evaluation(
            intervention, ood_transform, smoke=False
        )
        evaluation_records.append({
            'intervention': intervention,
            'ood_transform': ood_transform,
            'output_directory': str(output_dir),
            'run_summary': str(output_dir / 'run_summary.json'),
            'id_task_accuracy': summary['splits']['id_test']['task_accuracy'],
            'id_concept_accuracy': summary['splits']['id_test']['concept_accuracy'],
            'ood_auroc': (summary['ood_detection']['auroc'] if summary.get('ood_detection') else None),
        })
    evaluation_manifest = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'git_commit': subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip(),
        'dataset_sha256': sha256(Path(KAND_ZIP)),
        'checkpoints': CHECKPOINT_PATHS,
        'runs': evaluation_records,
    }
    manifest_path = Path(DRIVE_RESULTS_DIR) / 'evaluations' / f"evaluation_manifest_{time.strftime('%Y%m%d_%H%M%S')}.json"
    manifest_path.parent.mkdir(parents=True, exist_ok=True)
    manifest_path.write_text(json.dumps(evaluation_manifest, indent=2))
    print('Full evaluation complete. Manifest:', manifest_path)
else:
    print('Full MetaBEARS evaluation disabled.')


## Completion gate

A complete run contains the three baseline checkpoints plus `run_summary.json` artifacts for `figure_permute` and `palette_cycle`. The figure-permutation run also evaluates familiarity on a deterministic desaturated-palette OOD split.
